# 서울 에어비앤비 RevPAR 최적화 — Executive Summary

**RevPAR 구조 재정의 → 점유율 중심 전략 전환 → 실행 가능한 대시보드 구현**

---

## 핵심 결론 (3줄 요약)

1. **RevPAR 차이의 58.2%는 점유율로 설명** (가격(ADR)은 37.5%, log RevPAR 분산 분해)  
2. **가격 인하 ≠ 점유율 상승** — 두 변수는 독립적 관계  
3. **전략 전환 필요:** 가격 최적화보다 점유율 방어가 RevPAR에 더 직접적 영향

---

## 분석 흐름

```
데이터 수집 → EDA & 가설 검증 → RevPAR 구조 분석
    ↓
ADR / 점유율 모델링 (LightGBM)
    ↓
시장 군집화 (K-Means, k=4)
    ↓
RevPAR 시뮬레이션 → 전략 대시보드
```


## KPI 벤치마크

| 지표 | 값 |
|------|-----|
| TTM RevPAR 중위값 (전체) | ₩8,868 |
| TTM RevPAR 중위값 (Active+Operating) | ₩47,850 |
| Active+Operating 비율 | 44.9% |
| Dormant 비율 (리뷰 있음 + 최근 90일 예약 없음) | 54.3% |

---

## 가설 검증 결과

| 가설 | 결론 | 핵심 수치 |
|------|------|----------|
| H1: 슈퍼호스트 프리미엄 | ✅ 채택 | +92.3% RevPAR (중위값) |
| H2: 숙소 유형별 RevPAR 차이 | ✅ 채택 | entire_home >> private_room (2-3배) |
| H3: 사진·평점 정상관 | ⚠️ 부분 채택 | 정상관 확인, 최적 구간 없음 (사진 많을수록 RevPAR↑) |
| H4: min_nights 2-3박 최적 | ✅ 채택 | 1박·7박+ 대비 RevPAR 우위 |
| H5: 추가요금 정책 효과 | ❌ 기각 | 부과 숙소 RevPAR 더 낮음 (₩34,973 vs ₩50,387) |
| H6: 공급 과잉 → RevPAR 압박 | ⚠️ 약한 지지 | 마포구 최다 공급, 약한 음의 상관 |
| H7: POI 근접성 효과 | ⚠️ 약한 지지 | 거리 증가 시 RevPAR 감소 경향 |
| H8: 인구 구조 × RevPAR | ✅ 채택 | 관광형 > 업무형 > 혼합형 > 주거형 |

In [ ]:
# RevPAR = ADR x 점유율 -> log RevPAR = log ADR + log 점유율
# log RevPAR 분산을 각 항의 분산으로 나눠 점유율·ADR이 RevPAR 차이를 얼마나 설명하는지 분해
import numpy as np
import pandas as pd

df = pd.read_csv("../data/processed/seoul_airbnb_features.csv")
ao = df[(df["refined_status"] == "Active") & (df["operation_status"] == "Operating")]
d = ao[(ao["ttm_revpar"] > 0) & (ao["ttm_occupancy"] > 0) & (ao["ttm_avg_rate"] > 0)]

log_revpar = np.log(d["ttm_revpar"])
var_total = log_revpar.var()
occ_share = np.log(d["ttm_occupancy"]).var() / var_total
adr_share = np.log(d["ttm_avg_rate"]).var() / var_total

print(f"점유율 기여: {occ_share:.1%}")
print(f"ADR 기여:    {adr_share:.1%}")
print(f"공분산·잔차: {1 - occ_share - adr_share:.1%}")

## 전략적 의사결정 프레임

### RevPAR 기여도 분해
- **점유율 기여:** 58.2% — Var(log 점유율) / Var(log RevPAR)  
- **ADR 기여:** 37.5% — Var(log ADR) / Var(log RevPAR)  
- **공분산·잔차:** 4.4%

### 군집별 전략 방향 (K-Means k=4, 05·06 노트북 기준)

| 군집 | 유형 | 핵심 전략 |
|------|------|----------|
| Cluster 0 | 핫플 수익형 (강남·종로·중구·마포·용산) | 고급화 유지 + 슈퍼호스트 비율 유지 |
| Cluster 1 | 프리미엄 비즈니스 (서초·송파·영등포·성동·노원·도봉·중랑) | 슈퍼호스트 전환 + 즉시예약 + Dormant 재활성화 |
| Cluster 2 | 로컬 주거형 (강동·강서·양천·은평·강북·성북·광진) | 품질·사진·평점 개선 |
| Cluster 3 | 가성비 신흥형 (관악·구로·금천·동작·동대문·서대문) | Dormant 정리 + 비수기 프로모션 |

---

## 핵심 변화

- **가격 인하 중심 전략 → 점유율 중심 전략 전환**
- **감 기반 운영 → 데이터 기반 의사결정 가능**

---

*분석 기간: 2024-10-01 ~ 2025-09-30 (TTM 12개월)*  
*대상: 서울 에어비앤비 Active+Operating 14,399개 리스팅*